# Predictive Modelling of Housing Prices Using Machine Learning
## NYC Rolling Sales Data — February 2025 to January 2026
### Master's Dissertation | Academia de Studii Economice din București

---
This notebook covers the full analytical pipeline:
1. Data Loading & Merging
2. Data Preprocessing
3. Feature Engineering
4. Model Training & Evaluation
5. Feature Importance Analysis
6. Results Visualisation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

print('Libraries loaded successfully.')

## 2. Load and Merge Data

In [ ]:
# Update these paths to where you saved your files
files = {
    'Manhattan':     'rollingsales_manhattan.xlsx',
    'Brooklyn':      'rollingsales_brooklyn.xlsx',
    'Queens':        'rollingsales_queens.xlsx',
    'Bronx':         'rollingsales_bronx.xlsx',
    'Staten Island': 'rollingsales_statenisland.xlsx'
}

dfs = []
for borough, filepath in files.items():
    temp = pd.read_excel(filepath, skiprows=4)
    dfs.append(temp)
    print(f'{borough}: {len(temp):,} records loaded')

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal raw dataset: {len(df):,} rows x {df.shape[1]} columns')

In [ ]:
# Preview the data
df.head(3)

In [ ]:
# Check column names and data types
df.dtypes

## 3. Data Preprocessing

In [ ]:
# 3.1 Check SALE PRICE distribution
print('SALE PRICE = 0:', (df['SALE PRICE'] == 0).sum(), f"({(df['SALE PRICE'] == 0).mean()*100:.1f}%)")
print('SALE PRICE < 10,000:', (df['SALE PRICE'] < 10000).sum())
print('SALE PRICE >= 10,000:', (df['SALE PRICE'] >= 10000).sum())

In [ ]:
# 3.2 Filter non-market transactions (price <= $10,000)
df = df[df['SALE PRICE'] > 10000].copy()
print(f'After price filter: {len(df):,} rows')

In [ ]:
# 3.3 Drop irrelevant columns
drop_cols = [
    'EASEMENT',               # 100% missing
    'ADDRESS',                # identifier, not predictive
    'APARTMENT NUMBER',       # mostly missing
    'BLOCK', 'LOT',           # property identifiers
    'TAX CLASS AT PRESENT',   # collinear with building class
    'TAX CLASS AT TIME OF SALE',
    'SALE DATE'               # used only for filtering
]
df.drop(columns=drop_cols, inplace=True)
print('Remaining columns:', list(df.columns))

In [ ]:
# 3.4 Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# 3.5 Drop rows with missing GROSS SQUARE FEET (key predictor)
df = df[df['GROSS SQUARE FEET'].notna() & (df['GROSS SQUARE FEET'] > 0)]
print(f'After GSF filter: {len(df):,} rows')

In [ ]:
# 3.6 Impute YEAR BUILT with borough median
df['YEAR BUILT'] = df['YEAR BUILT'].fillna(
    df.groupby('BOROUGH')['YEAR BUILT'].transform('median'))
df['YEAR BUILT'] = df['YEAR BUILT'].fillna(1950)
print('YEAR BUILT missing after imputation:', df['YEAR BUILT'].isnull().sum())

In [ ]:
# 3.7 Remove outliers (above 99th percentile)
p99 = df['SALE PRICE'].quantile(0.99)
print(f'99th percentile: ${p99:,.0f}')
df = df[df['SALE PRICE'] <= p99]
print(f'After outlier removal: {len(df):,} rows')

In [ ]:
# Visualise SALE PRICE distribution before and after log transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SALE PRICE'] / 1e6, bins=60, color='#1F3864', edgecolor='white', alpha=0.85)
axes[0].set_title('Sale Price Distribution (USD millions)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sale Price ($ millions)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log(df['SALE PRICE']), bins=60, color='#2E5090', edgecolor='white', alpha=0.85)
axes[1].set_title('Log-Transformed Sale Price Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Sale Price)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('sale_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 4. Feature Engineering

In [ ]:
# 4.1 Create BUILDING AGE feature
df['BUILDING AGE'] = 2025 - df['YEAR BUILT']
print(f'Building Age — Min: {df["BUILDING AGE"].min():.0f}, Max: {df["BUILDING AGE"].max():.0f}, Median: {df["BUILDING AGE"].median():.0f}')

In [ ]:
# 4.2 Log-transform the target variable
df['LOG_SALE_PRICE'] = np.log(df['SALE PRICE'])
print('LOG_SALE_PRICE created.')

In [ ]:
# 4.3 Label encode categorical columns
cat_cols = ['NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'BUILDING CLASS AT TIME OF SALE']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f'{col}: encoded ({df[col].nunique()} unique values)')

In [ ]:
# 4.4 Define features and target
features = [
    'BOROUGH', 'NEIGHBORHOOD', 'BUILDING CLASS CATEGORY',
    'BUILDING CLASS AT TIME OF SALE', 'ZIP CODE',
    'RESIDENTIAL UNITS', 'TOTAL UNITS',
    'GROSS SQUARE FEET', 'BUILDING AGE'
]

X = df[features]
y = df['LOG_SALE_PRICE']
print(f'Features: {len(features)}')
print(f'Samples:  {len(X):,}')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr_df = df[features + ['LOG_SALE_PRICE']].corr()
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Matrix — Features vs Log Sale Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Training & Evaluation

In [ ]:
# 5.1 Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f'Training set:  {len(X_train):,} records')
print(f'Test set:      {len(X_test):,} records')

In [ ]:
# 5.2 Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_features='sqrt',
                                               random_state=42, n_jobs=-1),
    'XGBoost':           xgb.XGBRegressor(n_estimators=300, learning_rate=0.05,
                                           max_depth=6, subsample=0.8,
                                           random_state=42, verbosity=0)
}

In [ ]:
# 5.3 Train all models and collect metrics
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Convert back from log scale for interpretable metrics
    preds_exp   = np.exp(preds)
    y_test_exp  = np.exp(y_test)

    rmse = np.sqrt(mean_squared_error(y_test_exp, preds_exp))
    mae  = mean_absolute_error(y_test_exp, preds_exp)
    r2   = r2_score(y_test, preds)   # R² on log scale

    results[name] = {'RMSE ($)': rmse, 'MAE ($)': mae, 'R²': r2}
    trained_models[name] = model
    print(f'{name:20s} → RMSE: ${rmse:>12,.0f} | MAE: ${mae:>10,.0f} | R²: {r2:.4f}')

print('\nAll models trained.')

In [ ]:
# 5.4 Results table
results_df = pd.DataFrame(results).T.round(4)
results_df['RMSE ($)'] = results_df['RMSE ($)'].apply(lambda x: f'${x:,.0f}')
results_df['MAE ($)']  = results_df['MAE ($)'].apply(lambda x: f'${x:,.0f}')
print(results_df.to_string())

In [ ]:
# 5.5 Visualise model comparison (R²)
r2_vals  = {k: v['R²'] for k, v in results.items()}
colors = ['#d9534f' if v == max(r2_vals.values()) else '#2E5090' for v in r2_vals.values()]

plt.figure(figsize=(9, 5))
bars = plt.bar(r2_vals.keys(), r2_vals.values(), color=colors, edgecolor='white', width=0.5)
plt.ylim(0, 0.75)
plt.ylabel('R² Score', fontsize=12)
plt.title('Model Comparison — R² Score', fontsize=13, fontweight='bold')
for bar, val in zip(bars, r2_vals.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison_r2.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# 6.1 Random Forest feature importance
rf_model = trained_models['Random Forest']
importances_rf = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances_rf.plot(kind='barh', color='#1F3864', edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6.2 XGBoost feature importance
xgb_model = trained_models['XGBoost']
importances_xgb = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances_xgb.plot(kind='barh', color='#2E5090', edgecolor='white')
plt.title('Feature Importance — XGBoost', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 Actual vs Predicted — XGBoost
best_model = trained_models['XGBoost']
preds_best = np.exp(best_model.predict(X_test))
actual     = np.exp(y_test)

plt.figure(figsize=(8, 7))
plt.scatter(actual / 1e6, preds_best / 1e6, alpha=0.3, s=10, color='#2E5090')
max_val = max(actual.max(), preds_best.max()) / 1e6
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
plt.xlabel('Actual Sale Price ($ millions)', fontsize=12)
plt.ylabel('Predicted Sale Price ($ millions)', fontsize=12)
plt.title('Actual vs Predicted Sale Price — XGBoost', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted_xgboost.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== FINAL RESULTS SUMMARY ===')
for name, metrics in results.items():
    print(f"{name:20s} → RMSE: {metrics['RMSE ($)']:>14,.0f} | MAE: {metrics['MAE ($)']:>12,.0f} | R²: {metrics['R²']:.4f}")